In [1]:
import pandas as pd
import numpy as np
import igraph as ig
import re
import os

# Survival - NON-CCI Cliques

In [2]:
# CCIs
print('CCIs')
crosstalkdf = pd.read_csv('/home/lnemati/pathway_crosstalk/results/survival/aggregate/all_pairs.csv')
crosstalkdf['motif'] = np.where(crosstalkdf['motif'].isin(['3_clique', '4_clique']), 'clique', 'other')
crosstalkdf = crosstalkdf.query('tissue != "Pan_Cancer"')

significant = crosstalkdf.query('pval_adj < 0.05')
significant = significant.query('not ((hr - se) < 1 and 1 < (hr + se))')

better = significant.copy()
better = better.query('hr_lfc_best > 0.')
better = better.query('c_diff_best > 0.')
better = better.query('(logrank_pval < logrank_pval1) and (logrank_pval < logrank_pval2)')

df = better.query('tissue != "Pan_Cancer"')
all_counts = crosstalkdf.query('tissue != "Pan_Cancer"').motif.value_counts().sort_values(ascending=False)

# Calculate probabilities
probs = (df.motif.value_counts() / all_counts).sort_values()

print('Univariate Probs.')
print(probs.sort_values())

# Read coariates analysis
import pandas as pd
from pathlib import Path
from scipy.stats import false_discovery_control

parent_dir = Path('/home/lnemati/pathway_crosstalk/results/survival/covariates')

# find all csv files
csv_files = parent_dir.glob('*.csv')

# read and combine
covdf = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

# there are a few NaNs (e.g. model did not converge), drop them
covdf = covdf.dropna()

# Adjust pvalues
covdf['lrt_pval_adj'] = false_discovery_control(covdf['lrt_pval'])
covdf['lrt_significant'] = covdf['lrt_pval_adj'] < 0.05

# Add covariate columns to better df
covdf = covdf[['interaction', 'tissue', 'aic_lower', 'lrt_significant', 'lrt_pval_adj', 'covariates_used']].set_index(['interaction', 'tissue'])
better = better.join(covdf, on=['interaction', 'tissue'])

cliques = better.query('tissue != "Pan_Cancer" and motif == "clique"')
multi_probs = (better.query('aic_lower').motif.value_counts() / all_counts).sort_values()

print('Multivariate Probs. (AIC)')
print(multi_probs.sort_values())

multi_probs = (better.query('lrt_significant').motif.value_counts() / all_counts).sort_values()

print('Multivariate Probs. (LRT)')
print(multi_probs.sort_values())

# ------------------------------------------------------------
print('Non-CCIs')

# NON-CCIs
crosstalkdf = pd.read_csv('/home/lnemati/pathway_crosstalk/results/survival/non_cci_cliques/aggregate/all_pairs.csv')
crosstalkdf['motif'] = 'clique'
crosstalkdf = crosstalkdf.query('tissue != "Pan_Cancer"')

significant = crosstalkdf.query('pval_adj < 0.05')
significant = significant.query('not ((hr - se) < 1 and 1 < (hr + se))')

better = significant.copy()
better = better.query('hr_lfc_best > 0.')
better = better.query('c_diff_best > 0.')
better = better.query('(logrank_pval < logrank_pval1) and (logrank_pval < logrank_pval2)')

df = better.query('tissue != "Pan_Cancer"')
all_counts = crosstalkdf.query('tissue != "Pan_Cancer"').motif.value_counts().sort_values(ascending=False)

# Calculate probabilities
probs = (df.motif.value_counts() / all_counts).sort_values()

print('Univariate Probs.')
print(probs.sort_values())

# Read coariates analysis
import pandas as pd
from pathlib import Path
from scipy.stats import false_discovery_control

parent_dir = Path('/home/lnemati/pathway_crosstalk/results/survival/non_cci_cliques/covariates')

# find all csv files
csv_files = parent_dir.glob('*.csv')

# read and combine
covdf = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

# there are a few NaNs (e.g. model did not converge), drop them
covdf = covdf.dropna()

# Adjust pvalues
covdf['lrt_pval_adj'] = false_discovery_control(covdf['lrt_pval'])
covdf['lrt_significant'] = covdf['lrt_pval_adj'] < 0.05

# Add covariate columns to better df
covdf = covdf[['interaction', 'tissue', 'aic_lower', 'lrt_significant', 'lrt_pval_adj', 'covariates_used']].set_index(['interaction', 'tissue'])
better = better.join(covdf, on=['interaction', 'tissue'])

cliques = better.query('tissue != "Pan_Cancer" and motif == "clique"')
multi_probs = (better.query('aic_lower').motif.value_counts() / all_counts).sort_values()

print('Multivariate Probs. (AIC)')
print(multi_probs.sort_values())

multi_probs = (better.query('lrt_significant').motif.value_counts() / all_counts).sort_values()

print('Multivariate Probs. (LRT)')
print(multi_probs.sort_values())

CCIs
Univariate Probs.
motif
other     0.028152
clique    0.032087
Name: count, dtype: float64
Multivariate Probs. (AIC)
motif
other     0.019517
clique    0.021220
Name: count, dtype: float64
Multivariate Probs. (LRT)
motif
other     0.017498
clique    0.018834
Name: count, dtype: float64
Non-CCIs
Univariate Probs.
motif
clique    0.02477
Name: count, dtype: float64
Multivariate Probs. (AIC)
motif
clique    0.017479
Name: count, dtype: float64
Multivariate Probs. (LRT)
motif
clique    0.015204
Name: count, dtype: float64


In [81]:
from scipy.stats import fisher_exact

print('\n=== Fisher Tests (CCI vs Non-CCI) ===')

# ---------- Helper ----------
def fisher_from_counts(better_count, total_count, better_noncci, total_noncci, label):
    a = better_count
    b = total_count - better_count
    c = better_noncci
    d = total_noncci - better_noncci

    table = [[a, b], [c, d]]
    oddsratio, pval = fisher_exact(table)

    print(f'\n{label}')
    print('Contingency table:', table)
    print(f'Fraction CCI: {a/total_count:.6f}')
    print(f'Fraction Non-CCI: {c/total_noncci:.6f}')
    print(f'Fisher p-value: {pval:.6e}')
    
# totals
total_cci = len(crosstalkdf)

# univariate
better_cci = len(better)

# AIC
better_cci_aic = len(better.query('aic_lower'))

# LRT
better_cci_lrt = len(better.query('lrt_significant'))


=== Fisher Tests (CCI vs Non-CCI) ===


In [85]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import false_discovery_control, fisher_exact

print('CCIs (CLIQUES ONLY)')

# ============================================================
# CCI DATA
# ============================================================
crosstalkdf = pd.read_csv('/home/lnemati/pathway_crosstalk/results/survival/aggregate/all_pairs.csv')

crosstalkdf['motif'] = np.where(
    crosstalkdf['motif'].isin(['3_clique', '4_clique']),
    'clique',
    'other'
)

crosstalkdf = crosstalkdf.query('tissue != "Pan_Cancer"')

# keep ONLY cliques
crosstalkdf = crosstalkdf.query('motif == "clique"')

significant = crosstalkdf.query('pval_adj < 0.05')
significant = significant.query('not ((hr - se) < 1 and 1 < (hr + se))')

better = significant.copy()
better = better.query('hr_lfc_best > 0.')
better = better.query('c_diff_best > 0.')
better = better.query('(logrank_pval < logrank_pval1) and (logrank_pval < logrank_pval2)')

df = better

# IMPORTANT: clique-only margin (your requested "all_counts")
all_counts = crosstalkdf.motif.value_counts()

probs = (df.motif.value_counts() / all_counts)
print('Univariate Probs (CCI, cliques only).')
print(probs)

# ============================================================
# COVARIATES
# ============================================================
parent_dir = Path('/home/lnemati/pathway_crosstalk/results/survival/covariates')
covdf = pd.concat((pd.read_csv(file) for file in parent_dir.glob('*.csv')), ignore_index=True)
covdf = covdf.dropna()

covdf['lrt_pval_adj'] = false_discovery_control(covdf['lrt_pval'])
covdf['lrt_significant'] = covdf['lrt_pval_adj'] < 0.05

covdf = covdf[
    ['interaction', 'tissue', 'aic_lower', 'lrt_significant']
].set_index(['interaction', 'tissue'])

better = better.join(covdf, on=['interaction', 'tissue'])

# multivariate probs (still clique-only universe)
multi_aic = better.query('aic_lower')
multi_lrt = better.query('lrt_significant')

print('\nMultivariate Probs (AIC)')
print((multi_aic.motif.value_counts() / all_counts))

print('\nMultivariate Probs (LRT)')
print((multi_lrt.motif.value_counts() / all_counts))

# ============================================================
# STORE COUNTS (CCI)
# ============================================================
total_cci = len(crosstalkdf)
better_cci = len(better)
better_cci_aic = len(multi_aic)
better_cci_lrt = len(multi_lrt)


# ============================================================
# NON-CCI DATA
# ============================================================
print('\nNon-CCIs (CLIQUES ONLY)')

crosstalkdf = pd.read_csv('/home/lnemati/pathway_crosstalk/results/survival/non_cci_cliques/aggregate/all_pairs.csv')

crosstalkdf['motif'] = 'clique'
crosstalkdf = crosstalkdf.query('tissue != "Pan_Cancer"')

# already clique-only, so margin is clean
crosstalkdf = crosstalkdf.query('motif == "clique"')

significant = crosstalkdf.query('pval_adj < 0.05')
significant = significant.query('not ((hr - se) < 1 and 1 < (hr + se))')

better = significant.copy()
better = better.query('hr_lfc_best > 0.')
better = better.query('c_diff_best > 0.')
better = better.query('(logrank_pval < logrank_pval1) and (logrank_pval < logrank_pval2)')

df = better

all_counts_non = crosstalkdf.motif.value_counts()

probs = (df.motif.value_counts() / all_counts_non)
print('Univariate Probs (Non-CCI, cliques only).')
print(probs)

# covariates
parent_dir = Path('/home/lnemati/pathway_crosstalk/results/survival/non_cci_cliques/covariates')
covdf = pd.concat((pd.read_csv(file) for file in parent_dir.glob('*.csv')), ignore_index=True)
covdf = covdf.dropna()

covdf['lrt_pval_adj'] = false_discovery_control(covdf['lrt_pval'])
covdf['lrt_significant'] = covdf['lrt_pval_adj'] < 0.05

covdf = covdf[
    ['interaction', 'tissue', 'aic_lower', 'lrt_significant']
].set_index(['interaction', 'tissue'])

better = better.join(covdf, on=['interaction', 'tissue'])

multi_aic = better.query('aic_lower')
multi_lrt = better.query('lrt_significant')

print('\nMultivariate Probs (AIC)')
print((multi_aic.motif.value_counts() / all_counts_non))

print('\nMultivariate Probs (LRT)')
print((multi_lrt.motif.value_counts() / all_counts_non))

# ============================================================
# STORE COUNTS (NON-CCI)
# ============================================================
total_noncci = len(crosstalkdf)
better_noncci = len(better)
better_noncci_aic = len(multi_aic)
better_noncci_lrt = len(multi_lrt)


# ============================================================
# FISHER TESTS (CLIQUES ONLY, CONSISTENT MARGINS)
# ============================================================
print('\n=== Fisher Tests (CLIQUES ONLY) ===')

def fisher(a, na, b, nb, label):
    table = [[a, na - a],
             [b, nb - b]]
    _, p = fisher_exact(table)

    print(f'\n{label}')
    print('CCI fraction:', a / na)
    print('Non-CCI fraction:', b / nb)
    print('p-value:', p)

# Univariate
fisher(better_cci, total_cci,
       better_noncci, total_noncci,
       'Univariate')

# AIC
fisher(better_cci_aic, total_cci,
       better_noncci_aic, total_noncci,
       'Multivariate (AIC)')

# LRT
fisher(better_cci_lrt, total_cci,
       better_noncci_lrt, total_noncci,
       'Multivariate (LRT)')

CCIs (CLIQUES ONLY)
Univariate Probs (CCI, cliques only).
motif
clique    0.032087
Name: count, dtype: float64

Multivariate Probs (AIC)
motif
clique    0.02122
Name: count, dtype: float64

Multivariate Probs (LRT)
motif
clique    0.018834
Name: count, dtype: float64

Non-CCIs (CLIQUES ONLY)
Univariate Probs (Non-CCI, cliques only).
motif
clique    0.02477
Name: count, dtype: float64

Multivariate Probs (AIC)
motif
clique    0.017479
Name: count, dtype: float64

Multivariate Probs (LRT)
motif
clique    0.015204
Name: count, dtype: float64

=== Fisher Tests (CLIQUES ONLY) ===

Univariate
CCI fraction: 0.032087183323560926
Non-CCI fraction: 0.024770480648197868
p-value: 5.56596763228086e-41

Multivariate (AIC)
CCI fraction: 0.02121959602903415
Non-CCI fraction: 0.017478906829165426
p-value: 1.818814490605129e-16

Multivariate (LRT)
CCI fraction: 0.018833781516003154
Non-CCI fraction: 0.015203822749971195
p-value: 1.5343385608082435e-17


In [15]:
from scipy.stats import false_discovery_control

pvals = [5.56596763228086e-41, 1.818814490605129e-16, 1.5343385608082435e-17]
pvals_adj = false_discovery_control(pvals)
pvals_adj

array([1.66979029e-40, 1.81881449e-16, 2.30150784e-17])

In [48]:
tot_combinations = cliques.shape[0] # clique-tissue combinations
tot_combinations

22788

In [49]:
tot_cliques = cliques.interaction.nunique() # unique cliques
tot_cliques

17983

In [50]:
# N and Fraction clique-tissue combination with at least 1 tissue AIC lower
print(cliques.query('aic_lower').shape[0])
print(cliques.query('aic_lower').shape[0] / tot_combinations)

16080
0.7056345444971037


In [52]:
# N and Fraction clique with at least 1 tissue AIC lower
print(cliques.query('aic_lower').interaction.nunique())
print(cliques.query('aic_lower').interaction.nunique() / tot_cliques)

13863
0.7708947339153646


In [53]:
# N and Fraction clique-tissue combination with at least 1 tissue LRT significant
print(cliques.query('lrt_significant').shape[0])
print(cliques.query('lrt_significant').shape[0] / tot_combinations)

13987
0.6137879585746885


In [54]:
# N and Fraction clique with at least 1 tissue LRT significant
print(cliques.query('lrt_significant').interaction.nunique())
print(cliques.query('lrt_significant').interaction.nunique() / tot_cliques)

12436
0.6915420119001279


# Immunotherapy

In [16]:
import os
import pandas as pd
from scipy.stats import fisher_exact

# =========================
# Helper to load + filter
# =========================
def load_dataset(parentdir, motif_filter):
    dfs = []

    for tissue in os.listdir(parentdir):
        path = os.path.join(parentdir, tissue, 'aggregated', 'aggregated.csv')

        if not os.path.exists(path):
            continue

        tissuedf = pd.read_csv(path)
        tissuedf = tissuedf.rename(columns={'Unnamed: 0': 'interaction'})
        dfs.append(tissuedf)

    df = pd.concat(dfs, ignore_index=True)

    df = df.sort_values(by='auroc', ascending=False)
    df = df.query('motif != "random_pairs"')
    df = df.query(f'motif in {motif_filter}')

    return df


# =========================
# Load both datasets
# =========================
cci_parent = '/home/lnemati/pathway_crosstalk/results/immunotherapy/individual_interactions_and_motifs'
noncci_parent = '/home/lnemati/pathway_crosstalk/results/immunotherapy/non_cci_cliques/individual_interactions_and_motifs'

cci_df = load_dataset(cci_parent, ["3_clique", "4_clique"])
noncci_df = load_dataset(noncci_parent, ["3_non_cci_clique", "4_non_cci_clique"])


# =========================
# Function to compute stats
# =========================
def compute_fraction(df, condition):
    den = df.shape[0]
    num = df.query(condition).shape[0]
    return num, den, num / den if den > 0 else float('nan')


def compute_counts(df, condition):
    better = df.query(condition).shape[0]
    worse = df.shape[0] - better
    return better, worse


# =========================
# Conditions
# =========================
conds = {
    "AUROC": "auroc > auroc1 and auroc > auroc2",
    "AUPRC": "auprc > auprc1 and auprc > auprc2",
    "BOTH":  "auroc > auroc1 and auroc > auroc2 and auprc > auprc1 and auprc > auprc2"
}


# =========================
# Print original fractions
# =========================
print("=== FRACTIONS ===")

for name, cond in conds.items():
    print(f"\n{name}")

    num, den, frac = compute_fraction(cci_df, cond)
    print(f"CCI:     {num}/{den} = {frac:.4f}")

    num, den, frac = compute_fraction(noncci_df, cond)
    print(f"non-CCI: {num}/{den} = {frac:.4f}")


# =========================
# Fisher exact tests
# =========================
print("\n=== FISHER EXACT TESTS ===")

for name, cond in conds.items():
    a, b = compute_counts(cci_df, cond)
    c, d = compute_counts(noncci_df, cond)

    table = [[a, b], [c, d]]
    oddsratio, pvalue = fisher_exact(table)

    print(f"\n{name}")
    print(f"Contingency table: [[CCI better={a}, worse={b}], [nonCCI better={c}, worse={d}]]")
    print(f"Odds ratio: {oddsratio:.4f}")
    print(f"P-value: {pvalue:.4e}")

=== FRACTIONS ===

AUROC
CCI:     13399/38745 = 0.3458
non-CCI: 21171/69854 = 0.3031

AUPRC
CCI:     13840/38745 = 0.3572
non-CCI: 22180/69854 = 0.3175

BOTH
CCI:     10534/38745 = 0.2719
non-CCI: 16360/69854 = 0.2342

=== FISHER EXACT TESTS ===

AUROC
Contingency table: [[CCI better=13399, worse=25346], [nonCCI better=21171, worse=48683]]
Odds ratio: 1.2156
P-value: 2.9333e-47

AUPRC
Contingency table: [[CCI better=13840, worse=24905], [nonCCI better=22180, worse=47674]]
Odds ratio: 1.1945
P-value: 3.4933e-40

BOTH
Contingency table: [[CCI better=10534, worse=28211], [nonCCI better=16360, worse=53494]]
Odds ratio: 1.2209
P-value: 7.7965e-43


In [17]:
from scipy.stats import false_discovery_control

pvals = [2.9333e-47, 3.4933e-40, 7.7965e-43]
pvals_adj = false_discovery_control(pvals)
pvals_adj

array([8.799900e-47, 3.493300e-40, 1.169475e-42])

In [1]:
import pandas as pd
import numpy as np

In [4]:
# CCIs

## Read the results
parentdir = '/home/lnemati/pathway_crosstalk/results/immunotherapy/individual_interactions_and_motifs'

dfs = []

for tissue in os.listdir(parentdir):
    #if tissue in ['full_dataset']: # add back lymph and ureter when they finish running
    #    continue
    path = os.path.join(parentdir, tissue, 'aggregated', 'aggregated.csv')
    tissuedf = pd.read_csv(os.path.join(path))
    tissuedf = tissuedf.rename(columns={'Unnamed: 0': 'interaction'})
    #tissuedf['motif'] = tissuedf['motif'].replace({'3_clique': 'cliques', '4_clique': 'cliques'})

    dfs.append(tissuedf)
        
crosstalkdf = pd.concat(dfs, ignore_index=True)
#crosstalkdf = crosstalkdf.dropna()

crosstalkdf = crosstalkdf.sort_values(by='auroc')[::-1]

crosstalkdf = crosstalkdf.query('motif != "random_pairs"')
#crosstalkdf = crosstalkdf.query('tissue != "full_dataset"')

better = crosstalkdf.query('auroc > auroc1 and auroc > auroc2')
better = better.query('auprc > auprc1 and auprc > auprc2')

crosstalkdf = crosstalkdf.query('motif in ["3_clique", "4_clique"]')
den = crosstalkdf.shape[0]

print('AUROC')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2').shape[0]
print(num / den)

print('AUPRC')

num = crosstalkdf.query('auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)

print('BOTH')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2 and auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)


In [19]:
# NON-CCIs

## Read the results
parentdir = '/home/lnemati/pathway_crosstalk/results/immunotherapy/non_cci_cliques/individual_interactions_and_motifs'

dfs = []

for tissue in os.listdir(parentdir):
    #if tissue in ['full_dataset']: # add back lymph and ureter when they finish running
    #    continue
    path = os.path.join(parentdir, tissue, 'aggregated', 'aggregated.csv')
    if not os.path.exists(os.path.join(path)):
        continue
    tissuedf = pd.read_csv(os.path.join(path))
    tissuedf = tissuedf.rename(columns={'Unnamed: 0': 'interaction'})
    #tissuedf['motif'] = tissuedf['motif'].replace({'3_clique': 'cliques', '4_clique': 'cliques'})

    dfs.append(tissuedf)
        
crosstalkdf = pd.concat(dfs, ignore_index=True)
#crosstalkdf = crosstalkdf.dropna()

crosstalkdf = crosstalkdf.sort_values(by='auroc')[::-1]

crosstalkdf = crosstalkdf.query('motif != "random_pairs"')
#crosstalkdf = crosstalkdf.query('tissue != "full_dataset"')

better = crosstalkdf.query('auroc > auroc1 and auroc > auroc2')
better = better.query('auprc > auprc1 and auprc > auprc2')

crosstalkdf = crosstalkdf.query('motif in ["3_non_cci_clique", "4_non_cci_clique"]')
den = crosstalkdf.shape[0]

print('AUROC')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2').shape[0]
print(num / den)

print('AUPRC')

num = crosstalkdf.query('auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)

print('BOTH')

num = crosstalkdf.query('auroc > auroc1 and auroc > auroc2 and auprc > auprc1 and auprc > auprc2').shape[0]
print(num / den)


AUROC
0.3030749849686489
AUPRC
0.3175193976007101
BOTH
0.23420276576860308


# Get NON-CCI cliques

In [18]:
network = pd.read_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/network_filtered.csv')
network = network.query('auroc > 0.5')
network = network[~network['ccc']]

In [19]:
import pandas as pd
import igraph as ig

# assume your dataframe is called `network`

# 1. Build edge list
edges = list(zip(network["complex1"], network["complex2"]))

# 2. Get unique node names
nodes = list(set(network["complex1"]).union(set(network["complex2"])))

# 3. Create graph
g = ig.Graph()
g.add_vertices(nodes)
g.add_edges(edges)

# Ensure it's undirected and unweighted
g = g.as_undirected()
g.simplify()  # removes duplicates / self-loops if any

In [20]:
# 4. Find cliques
cliques_3 = g.cliques(min=3, max=3)
cliques_4 = g.cliques(min=4, max=4)

# 5. Convert vertex indices back to names
cliques_3_named = [[g.vs[idx]["name"] for idx in clique] for clique in cliques_3]
cliques_4_named = [[g.vs[idx]["name"] for idx in clique] for clique in cliques_4]

print("3-node cliques:", cliques_3_named[:5])
print("4-node cliques:", cliques_4_named[:5])

3-node cliques: [['ACVR2A_BMPR1A', 'CD24', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'IL20RA_IL20RB_IL2RG', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'ERBB3', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'DAGLA', 'ACVR1_BMPR2'], ['ACVR2A_BMPR1A', 'ACVR1_BMPR2', 'CEL']]
4-node cliques: [['ACVR1_ACVR2A', 'CD8A_CD8B', 'PRLR', 'ENPP4'], ['ACVR1_ACVR2A', 'IL15RA_IL2RB_IL2RG', 'CD8A_CD8B', 'PRLR'], ['ACVR1_ACVR2A', 'IL21R_IL2RG', 'CD8A_CD8B', 'PRLR'], ['ACVR1_ACVR2A', 'CD8A_CD8B', 'IL2RA_IL2RB_IL2RG', 'PRLR'], ['CD8A_CD8B', 'ACVR2A_BMPR1A', 'IL2RA_IL2RB_IL2RG', 'EFNA5']]


In [47]:
487 + 3818

4305

In [48]:
487 / 3818

0.12755369303300157

In [39]:
import random
import pandas as pd

N3 = 487
N4 = 3818
factor = 10

# sample cliques
sample_3 = random.sample(cliques_3_named, min(len(cliques_3_named), factor * N3))
sample_4 = random.sample(cliques_4_named, min(len(cliques_4_named), factor * N4))

# --- 3-node cliques ---
interactions = []
for clique in sample_3:
    shared = random.choice(clique)
    others = [x for x in clique if x != shared]
    int1 = f"{shared}+{others[0]}"
    int2 = f"{shared}+{others[1]}"
    interactions.append(f"{int1}&{int2}")

df3 = pd.DataFrame(interactions, columns=["Interaction"])
df3["Type"] = "3_non_cci_clique"


# --- 4-node cliques ---
interactions = []
for clique in sample_4:
    pair1 = random.sample(clique, 2)
    pair2 = [x for x in clique if x not in pair1]
    int1 = f"{pair1[0]}+{pair1[1]}"
    int2 = f"{pair2[0]}+{pair2[1]}"
    interactions.append(f"{int1}&{int2}")

df4 = pd.DataFrame(interactions, columns=["Interaction"])
df4["Type"] = "4_non_cci_clique"

In [42]:
non_cci_cliques = pd.concat([df3, df4])

In [44]:
non_cci_cliques.to_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/non_cci_cliques/motifs.csv')

In [4]:
non_cci_cliques = pd.read_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/non_cci_cliques/motifs.csv', index_col=0)
non_cci_cliques

,Interaction,Type
0,COL22A1+NRP1_PLXNA1&COL22A1+STRA6,3_non_cci_clique
1,ULBP3+CD8A_CD8B&ULBP3+ITGA4_ITGB7,3_non_cci_clique
2,TNFSF4+CD200R1&TNFSF4+CD28,3_non_cci_clique
3,HRH1+NECTIN4&HRH1+PTHLH,3_non_cci_clique
4,F2RL2+IL24&F2RL2+ENPP1,3_non_cci_clique
...,...,...
38175,LILRB3+CD28&CD2+ADA2,4_non_cci_clique
38176,ITGAX_ITGB2+FPR1&TNFSF10+TREM2_TYROBP,4_non_cci_clique
38177,SIGLEC10+P2RY6&CD48+COL12A1,4_non_cci_clique
38178,TEK+ACVR1_ACVR2A&POSTN+THY1,4_non_cci_clique


In [5]:
non_cci_cliques_reduced = non_cci_cliques.iloc[::5]
#non_cci_cliques_reduced.to_csv('/home/lnemati/pathway_crosstalk/results/crosstalk/all_ccc_complex_pairs/adj/non_cci_cliques/motifs_reduced_for_immunotherapy.csv')

In [8]:
pseudo_ccc = pd.DataFrame(list(set(non_cci_cliques_reduced.Interaction.str.split('&').sum())), columns=['interaction'])
pseudo_ccc['complex_a'] = pseudo_ccc['interaction'].str.split('+', expand=True)[0]
pseudo_ccc['complex_b'] = pseudo_ccc['interaction'].str.split('+', expand=True)[1]
pseudo_ccc['all_genes'] = pseudo_ccc['interaction'].apply(lambda x: sorted(re.split('[+_]', x)))
keep = pseudo_ccc['all_genes'].drop_duplicates().index
pseudo_ccc = pseudo_ccc.loc[keep].reset_index()
#pseudo_ccc.to_csv('/home/lnemati/pathway_crosstalk/data/interactions/pseudo_ccc_reduced_from_non_cci_cliques.csv')

In [7]:
pseudo_ccc

,index,interaction,complex_a,complex_b,all_genes
0,0,LCK+TNFSF8,LCK,TNFSF8,"[LCK, TNFSF8]"
1,1,ANGPTL4+COL7A1,ANGPTL4,COL7A1,"[ANGPTL4, COL7A1]"
2,2,IL15RA_IL2RB_IL2RG+UBASH3B,IL15RA_IL2RB_IL2RG,UBASH3B,"[IL15RA, IL2RB, IL2RG, UBASH3B]"
3,3,PILRA+P2RY6,PILRA,P2RY6,"[P2RY6, PILRA]"
4,4,SEMA3G+CYSLTR1,SEMA3G,CYSLTR1,"[CYSLTR1, SEMA3G]"
...,...,...,...,...,...
13134,14484,CXCR4+LILRB1,CXCR4,LILRB1,"[CXCR4, LILRB1]"
13135,14486,ITGA4_ITGB1+P2RY13,ITGA4_ITGB1,P2RY13,"[ITGA4, ITGB1, P2RY13]"
13136,14487,HCST_KLRK1+CD72,HCST_KLRK1,CD72,"[CD72, HCST, KLRK1]"
13137,14489,CDH11+TNFSF8,CDH11,TNFSF8,"[CDH11, TNFSF8]"


In [30]:
pseudo_ccc

,index,interaction,complex_a,complex_b,all_genes
0,0,NT5E_SLC29A1+CXCL10,NT5E_SLC29A1,CXCL10,"[CXCL10, NT5E, SLC29A1]"
1,1,TNFSF4+GPR34,TNFSF4,GPR34,"[GPR34, TNFSF4]"
2,2,CXCR6+TREM2_TYROBP,CXCR6,TREM2_TYROBP,"[CXCR6, TREM2, TYROBP]"
3,3,SIRPG+BOC_PTCH1,SIRPG,BOC_PTCH1,"[BOC, PTCH1, SIRPG]"
4,4,S1PR1+SIRPG,S1PR1,SIRPG,"[S1PR1, SIRPG]"
...,...,...,...,...,...
34005,45879,TNFRSF18+TNFSF4,TNFRSF18,TNFSF4,"[TNFRSF18, TNFSF4]"
34006,45880,ENPP1+ACVR2A_BMPR1A,ENPP1,ACVR2A_BMPR1A,"[ACVR2A, BMPR1A, ENPP1]"
34007,45882,IL1B+FLT3LG,IL1B,FLT3LG,"[FLT3LG, IL1B]"
34008,45883,EPHA4+CCR7,EPHA4,CCR7,"[CCR7, EPHA4]"
